# Customer Churn Predictor — EDA & Feature Selection
**Introduction to Data Science | Final Project**

This notebook covers the full analytical pipeline:
1. Load raw GitHub user data from the saved JSON file
2. Exploratory Data Analysis
3. Feature generation (8 behavioral features)
4. Feature selection — all 4 methods (Filter, RFE, Decision Tree, Random Forest)
5. Comparison table with final keep/drop decisions
6. Train and save the final model to `app/model.pkl`

## 0 — Imports & Setup

In [ ]:
import sys, os
sys.path.insert(0, "../app")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from scraper import load_raw_data, fetch_all_users
from features import generate_features, get_X_y, FEATURE_COLUMNS, check_class_balance
from model import train, save_model
from dotenv import load_dotenv

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)
print("Imports OK")

## 1 — Load Raw Data

We load directly from the saved JSON file if no Github API is present in the `.env` file (not present in the repo).

The `load_raw_data()` function in `scraper.py` handles both plain list and wrapped JSON structures.

In case there is a Github Token then the `fetch_all_users` function is called also from `scraper.py`.

In [ ]:
df_raw = load_raw_data("../data/raw/users_raw.json")
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 2 — Exploratory Data Analysis

Before generating features, we inspect the raw data for missing values,
data types, and basic distributions.

In [ ]:
print("=== Data Types ===")
print(df_raw.dtypes)
print()
print("=== Null Counts ===")
print(df_raw.isnull().sum())
print()
print("=== Basic Stats ===")
df_raw.describe()

In [ ]:
# Distribution of raw numeric fields
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

columns_to_plot = ["prs_opened", "total_commits", "events_30d", "distinct_repos"]
for ax, col in zip(axes, columns_to_plot):
    ax.hist(df_raw[col].clip(upper=df_raw[col].quantile(0.95)), bins=30,
            color="steelblue", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("Value")

plt.suptitle("Raw Field Distributions (clipped at 95th percentile)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Activity dates fill rate
# Check if the user has any activity dates recorded in the array
has_activity_rate = (df_raw["activity_dates"].apply(lambda x: len(x) if isinstance(x, list) else 0) > 0).mean()
print(f"Users with recorded activity dates: {has_activity_rate:.1%}")
print(f"Users without activity dates: {1 - has_activity_rate:.1%}")

## 3 — Feature Generation

Raw API fields are not features. We transform them into 12 behavioral signals
that encode domain knowledge about GitHub engagement patterns.

| Feature | Type | Reasoning |
|---|---|---|
| `pr_success_rate` | Ratio | Quality/success of contributions (`prs_merged` / `prs_opened`) |
| `social_engagement_ratio` | Ratio | Community vs solo coding (`comments` / `commits + comments`) |
| `recency_gap` | Time-based | Overall days since last activity of ANY type |
| `recency_gap_push` | Time-based | Days since last push (code creation) |
| `recency_gap_pr` | Time-based | Days since last pull request (collaboration) |
| `recency_gap_issue` | Time-based | Days since last issue (project management) |
| `recency_gap_comment` | Time-based | Days since last comment (community engagement) |
| `max_inactivity_streak` | Time-based | Longest gap between consecutive events in the ~30d window |
| `recent_velocity` | Aggregation | Short-term engagement (`events_30d`) |
| `commit_velocity_1y` | Aggregation | Historical commit pace (`total_commits / 365`) |
| `velocity_drop` | Aggregation | Deceleration: historical pace minus recent pace |
| `repo_density` | Aggregation | Breadth of ecosystem involvement (`distinct_repos` / years) |
| `is_org_member` | Binary | Professional/community integration (`org_count > 0`) |
| `has_secure_profile` | Binary | Security hygiene (`ssh_key_count` + `gpg_key_count` > 0) |

**Note on the 30-day window:** GitHub's public events API returns at most the last ~30 days of activity. We therefore define churn as **no public events in the last 30 days**. This is a standard monthly-active-user churn definition and keeps the model honest about what the data actually observes.

In [ ]:
X, y = get_X_y(df_raw)
check_class_balance(y)
print(f"\nFeature matrix shape: {X.shape}")
X.describe()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 5, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLUMNS):
    axes[i].hist(X[col], bins=30, color="steelblue", edgecolor="white")
    axes[i].set_title(col, fontsize=11)

plt.suptitle("Generated Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Churn label distribution
fig, axes = plt.subplots(figsize=(5, 4))
y.value_counts().plot(kind="bar", axes=axes, color=["coral", "steelblue"], edgecolor="white")
axes.set_xticklabels([ "Churned","Retained"], rotation=0)
axes.set_ylabel("Count")
axes.set_title("Churn Label Distribution")
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

### Method 1 — Filter Methods

Filter methods evaluate each feature **independently of any model**.
They are fast and ideal for removing obviously useless features before the more expensive methods run.

Three filters are applied:
- **Variance Threshold** — removes features that barely change across users
- **Correlation Matrix** — flags redundant feature pairs (corr > 0.9)
- **ANOVA F-test** — measures how much each feature's distribution differs between churned vs retained

In [ ]:
# ── Variance Threshold ──────────────────────────────────────────────
sel_var = VarianceThreshold(threshold=0.01)
sel_var.fit(X_train)

low_var_features = X.columns[~sel_var.get_support()].tolist()
print(f"Features removed by variance threshold: {low_var_features or 'none — all features have sufficient variance'}")

In [ ]:
# ── Correlation Matrix ──────────────────────────────────────────────
corr = X.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

# Flag pairs above threshold
CORR_THRESHOLD = 0.9
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) > CORR_THRESHOLD:
            high_corr_pairs.append((corr.index[i], corr.columns[j], round(val, 3)))

print(f"Highly correlated pairs (|r| > {CORR_THRESHOLD}):")
for pair in high_corr_pairs:
    print(f"  {pair[0]} — {pair[1]}: r = {pair[2]}")
if not high_corr_pairs:
    print("  None found.")

In [ ]:
# ── ANOVA F-test ────────────────────────────────────────────────────
sel_k = SelectKBest(score_func=f_classif, k=7)
sel_k.fit(X_train, y_train)

filter_df = pd.DataFrame({
    "feature":      FEATURE_COLUMNS,
    "f_score":      sel_k.scores_,
    "p_value":      sel_k.pvalues_,
    "selected":     sel_k.get_support(),
}).sort_values("f_score", ascending=False).reset_index(drop=True)

filter_df["filter_rank"] = range(1, len(filter_df) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue" if s else "lightgray" for s in filter_df["selected"]]
ax.barh(filter_df["feature"][::-1], filter_df["f_score"][::-1], color=colors[::-1])
ax.set_xlabel("ANOVA F-score")
ax.set_title("Filter Method — ANOVA F-test (blue = selected top 5)")
plt.tight_layout()
plt.show()

print(filter_df.to_string(index=False))

### Method 2 — Wrapper: Recursive Feature Elimination (RFE)

RFE trains a Logistic Regression, removes the weakest feature (by coefficient magnitude),
retrains, and repeats until the desired feature count is reached.
Unlike filters, RFE captures interactions between features because it uses an actual model.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr, n_features_to_select=7, step=1)
rfe.fit(X_train, y_train)

rfe_df = pd.DataFrame({
    "feature":      FEATURE_COLUMNS,
    "rfe_selected": rfe.support_,
    "rfe_ranking":  rfe.ranking_,
}).sort_values("rfe_ranking").reset_index(drop=True)

print("RFE — Selected features:")
print(rfe_df[rfe_df["rfe_selected"]]["feature"].tolist())
print()
print(rfe_df.to_string(index=False))

### Method 3 — Decision Tree Feature Importance

A single Decision Tree scores features by how much they reduce Gini impurity at each split.
Features used near the root (early splits) get higher importance scores.

**Limitation:** a single tree is unstable — a different random seed or data split
can produce a very different ranking. Compare with Random Forest results.

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

dt_df = pd.DataFrame({
    "feature":       FEATURE_COLUMNS,
    "dt_importance": dt.feature_importances_,
}).sort_values("dt_importance", ascending=False).reset_index(drop=True)

dt_df["dt_rank"] = range(1, len(dt_df) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(dt_df["feature"][::-1], dt_df["dt_importance"][::-1], color="coral", edgecolor="white")
ax.set_xlabel("Gini Importance")
ax.set_title("Decision Tree Feature Importance")
plt.tight_layout()
plt.show()

print(dt_df.to_string(index=False))

### Method 4 — Random Forest Feature Importance

Random Forest trains 100 decision trees, each on a random bootstrap sample,
and **averages** their importance scores. This averaging makes the ranking
significantly more stable and trustworthy than a single Decision Tree.

If a feature ranks highly across all 100 trees, that's a robust signal.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_df = pd.DataFrame({
    "feature":      FEATURE_COLUMNS,
    "rf_importance": rf.feature_importances_,
}).sort_values("rf_importance", ascending=False).reset_index(drop=True)

rf_df["rf_rank"] = range(1, len(rf_df) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(rf_df["feature"][::-1], rf_df["rf_importance"][::-1], color="steelblue", edgecolor="white")
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_title("Random Forest Feature Importance (100 trees)")
plt.tight_layout()
plt.show()

print(rf_df.to_string(index=False))

## 5 — Comparison Table

We now synthesise results from all four methods into a single table.
Features that rank highly **across all methods** are the strongest candidates to keep.
Disagreements between methods are the most analytically interesting findings.

In [ ]:
comparison = pd.DataFrame({"feature": FEATURE_COLUMNS})

# Filter rank
comparison = comparison.merge(
    filter_df[["feature", "filter_rank"]], on="feature", how="left"
)

# RFE selected
comparison = comparison.merge(
    rfe_df[["feature", "rfe_selected"]], on="feature", how="left"
)
comparison["rfe_selected"] = comparison["rfe_selected"].map({True: "✅", False: "❌"})

# DT rank
comparison = comparison.merge(
    dt_df[["feature", "dt_rank"]], on="feature", how="left"
)

# RF rank
comparison = comparison.merge(
    rf_df[["feature", "rf_rank"]], on="feature", how="left"
)

# Decision logic
def decide(row):
    avg_rank = (row["filter_rank"] + row["dt_rank"] + row["rf_rank"]) / 3
    rfe_ok   = row["rfe_selected"] == "✅"
    if avg_rank <= 4 and rfe_ok:
        return "✅ Keep"
    elif avg_rank <= 5 or rfe_ok:
        return "⚠️  Optional"
    else:
        return "❌ Drop"

comparison["decision"] = comparison.apply(decide, axis=1)
comparison = comparison.sort_values("rf_rank").reset_index(drop=True)

print("=" * 72)
print("FEATURE SELECTION COMPARISON TABLE")
print("=" * 72)
comparison

In [ ]:
# Highlight agreements and disagreements
kept     = comparison[comparison["decision"] == "✅ Keep"]["feature"].tolist()
optional = comparison[comparison["decision"] == "⚠️  Optional"]["feature"].tolist()
dropped  = comparison[comparison["decision"] == "❌ Drop"]["feature"].tolist()

print(f"Keep ({len(kept)}):     {kept}")
print(f"Optional ({len(optional)}): {optional}")
print(f"Drop ({len(dropped)}):     {dropped}")

### Observations

Write your analysis here after running the cells above. Address these questions:

1. **Which feature ranked #1 across all methods?** What does that tell you about the primary driver of churn?
2. **Where did Filter and RFE disagree?** Filter methods ignore feature interactions; RFE captures them via the logistic regression model. A feature that's weak in isolation can gain power in combination.
3. **Where did DT and RF disagree?** If they conflict, trust RF — averaging 100 trees eliminates the variance of a single tree.
4. **Are any features highly correlated?** If so, which one did you drop and why?

## 6 — Train and Save Final Model

We now train on the **full dataset** (not just the train split) using the features
selected from the comparison table, then save to `app/model.pkl` for the Docker API.

In [ ]:
final_model = train(df_raw, save=True)
print("\nmodel.pkl saved — ready for docker-compose up")

In [ ]:
# Sanity check — simulate a /predict call end to end
from features import features_from_dict
from model import predict as model_predict

test_user = {
    "created_at":       "2018-01-01T00:00:00Z",
    "last_activity_at": "2022-01-01T00:00:00Z",
    "last_push_at":     "2022-01-01T00:00:00Z",
    "last_pr_at":       None,
    "last_issue_at":    None,
    "last_comment_at":  None,
    "prs_merged":       3,
    "prs_opened":       5,
    "issue_comments":   10,
    "pr_comments":      2,
    "total_commits":    100,
    "events_30d":       0,
    "distinct_repos":   4,
    "activity_dates":   [],
    "org_count":        1,
    "ssh_key_count":    1,
    "gpg_key_count":    0
}

feature_array = features_from_dict(test_user)
result = model_predict(final_model, feature_array)

print("Test user prediction:")
print(f"  Churned:           {result['churned']}")
print(f"  Churn probability: {result['churn_probability']}")
print()
print("If this matches your expectation (inactive user → high churn), the pipeline is working correctly.")